# DOOM — Build & Run (WebAssembly Edition)

This notebook walks through every step needed to compile the classic **linuxdoom-1.10** source to WebAssembly with Emscripten and serve it in a browser via a Next.js dev server.

| Step | What happens |
|------|-------------|
| 1 | `npm install` — install web-doom front-end dependencies |
| 2 | Install / activate the **Emscripten SDK** (`emcc`) |
| 3 | Compile all DOOM `.c` files → `doom.js` + `doom.wasm` + `doom.data` |
| 4 | Verify the build artefacts exist in `web-doom/public/` |
| 5 | Start the **Next.js dev server** and open the game in the browser |

> **Prerequisites:** Node ≥ 18, Git, and internet access (first run only).

## Step 1 — Install front-end dependencies

Runs `npm install` inside `web-doom/` to pull down Next.js and the rest of the React front-end packages.

In [1]:
import subprocess, sys, os

ROOT = r'c:\projects\DOOM'
WEB  = os.path.join(ROOT, 'web-doom')
DOOM = os.path.join(ROOT, 'linuxdoom-1.10')

def run(cmd, cwd, timeout=300, shell=False):
    print(f'>>> {cmd if isinstance(cmd,str) else " ".join(cmd)}')
    sys.stdout.flush()
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                       timeout=timeout, shell=shell)
    if r.stdout: print(r.stdout[-3000:])
    if r.stderr: print('STDERR:', r.stderr[-1000:])
    print(f'Exit: {r.returncode}')
    return r.returncode

print('Step 1: npm install')
rc = run('npm install', WEB, timeout=180, shell=True)
print('npm install exit code:', rc)


Step 1: npm install
>>> npm install



up to date, audited 22 packages in 6s

3 packages are looking for funding
  run `npm fund` for details

1 high severity vulnerability

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.

Exit: 0
npm install exit code: 0


## Step 2 — Install Emscripten SDK

Clones the [emsdk](https://github.com/emscripten-core/emsdk) repo (if not already present), installs the latest toolchain, and activates it so that `emcc` is available for the build step.  
This downloads ~200 MB on first run and can take a few minutes.

In [2]:
import shutil, pathlib, subprocess, sys

EMSDK_DIR = pathlib.Path.home() / 'emsdk'

def sh(cmd, cwd=None, timeout=600):
    print(f'>>> {cmd}')
    sys.stdout.flush()
    r = subprocess.run(cmd, cwd=cwd or str(EMSDK_DIR), capture_output=True,
                       text=True, timeout=timeout, shell=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
    print(f'Exit: {r.returncode}')
    return r.returncode == 0

print(f'Installing Emscripten SDK to {EMSDK_DIR} ...')

if not (EMSDK_DIR / 'emsdk.ps1').exists():
    ok = sh(f'git clone https://github.com/emscripten-core/emsdk.git "{EMSDK_DIR}"', cwd=str(pathlib.Path.home()), timeout=120)
    if not ok:
        print('git clone failed!'); raise SystemExit(1)
else:
    print('emsdk already cloned, pulling latest...')
    sh('git pull', timeout=60)

print('\nInstalling latest Emscripten (this downloads ~200 MB, takes a few minutes)...')
ok = sh('powershell -ExecutionPolicy Bypass .\\emsdk install latest', timeout=900)
if not ok:
    print('emsdk install failed!'); raise SystemExit(1)

print('\nActivating...')
ok = sh('powershell -ExecutionPolicy Bypass .\\emsdk activate latest', timeout=120)
if not ok:
    print('emsdk activate failed!'); raise SystemExit(1)

# Set PATH so subsequent subprocess calls find emcc
import os
upstream_bin = str(EMSDK_DIR / 'upstream' / 'emscripten')
node_bin     = str(list(EMSDK_DIR.glob('node/*/bin'))[0]) if list(EMSDK_DIR.glob('node/*/bin')) else ''
os.environ['PATH'] = upstream_bin + os.pathsep + node_bin + os.pathsep + os.environ.get('PATH','')
os.environ['EMSDK']     = str(EMSDK_DIR)
os.environ['EM_CONFIG'] = str(EMSDK_DIR / '.emscripten')

emcc = shutil.which('emcc')
print(f'\nemcc: {emcc}')
if emcc:
    print('Emscripten ready!')
else:
    print('WARNING: emcc still not in PATH - check emsdk install output above')


Installing Emscripten SDK to C:\Users\Surya VM\emsdk ...
emsdk already cloned, pulling latest...
>>> git pull
Already up to date.

Exit: 0

Installing latest Emscripten (this downloads ~200 MB, takes a few minutes)...
>>> powershell -ExecutionPolicy Bypass .\emsdk install latest
Resolving SDK alias 'latest' to '5.0.2'
Resolving SDK version '5.0.2' to 'sdk-releases-0a320d2395858e63288b3632b81535444ca2c59d-64bit'
Installing SDK 'sdk-releases-0a320d2395858e63288b3632b81535444ca2c59d-64bit'..
Skipped installing node-22.16.0-64bit, already installed.
Skipped installing python-3.13.3-64bit, already installed.
Skipped installing releases-0a320d2395858e63288b3632b81535444ca2c59d-64bit, already installed.
All SDK components already installed: 'sdk-releases-0a320d2395858e63288b3632b81535444ca2c59d-64bit'.

Exit: 0

Activating...
>>> powershell -ExecutionPolicy Bypass .\emsdk activate latest
Resolving SDK alias 'latest' to '5.0.2'
Resolving SDK version '5.0.2' to 'sdk-releases-0a320d2395858e63288

## Step 3 — Compile DOOM → WebAssembly

Runs `build_em.ps1` which invokes `emcc` on all DOOM `.c` source files (excluding the original platform back-ends) and bundles the WAD into the output.  
Artefacts land in `web-doom/public/`: `doom.js`, `doom.wasm`, `doom.data`, and `wad-info.json`.

In [3]:
import shutil, subprocess, sys
emcc = shutil.which('emcc')
print(f'emcc: {emcc}')

print('\nStep 2: Build DOOM -> WebAssembly')
r = subprocess.run(
    'powershell -ExecutionPolicy Bypass -File build_em.ps1',
    cwd=DOOM, capture_output=True, text=True, timeout=600, shell=True
)
# Write errors to files for inspection
with open(r'c:\projects\DOOM\build_stdout.txt', 'w') as f: f.write(r.stdout or '')
with open(r'c:\projects\DOOM\build_stderr.txt', 'w') as f: f.write(r.stderr or '')
print('Exit:', r.returncode)
print('Full output saved to build_stdout.txt / build_stderr.txt')

# Show first 2000 chars of stderr (where link errors appear)
err = (r.stderr or '')
# Find lines with 'error:'
error_lines = [l for l in err.splitlines() if 'error:' in l.lower() or 'undefined' in l.lower()]
print(f'\nError lines ({len(error_lines)} total):')
for l in error_lines[:30]:
    print(l)


emcc: C:\Users\Surya VM\emsdk\upstream\emscripten\emcc.BAT

Step 2: Build DOOM -> WebAssembly
Exit: 0
Full output saved to build_stdout.txt / build_stderr.txt

Error lines (0 total):


## Step 4 — Verify build artefacts

Quick sanity-check: confirms that all four expected files exist in `web-doom/public/` and prints their sizes.

In [4]:
# Verify build output
import pathlib
pub = pathlib.Path(ROOT) / 'web-doom' / 'public'
for f in ['doom.js', 'doom.wasm', 'doom.data', 'wad-info.json']:
    p = pub / f
    size = p.stat().st_size if p.exists() else None
    print(f'{f}: {"EXISTS " + str(size) + " bytes" if size else "MISSING"}')

doom.js: EXISTS 180973 bytes
doom.wasm: EXISTS 935634 bytes
doom.data: EXISTS 28795076 bytes
wad-info.json: EXISTS 53 bytes


## Step 5 — Start the dev server & play

Launches `npm run dev` inside `web-doom/` and waits until Next.js reports it is listening.  
Once ready, the cell below will open **http://localhost:3000** automatically.

> Click the canvas to capture the mouse, then use **WASD / arrow keys** to move and the mouse to aim.

In [ ]:
import subprocess, time, re

# Always use --prefix so npm is guaranteed to resolve web-doom/package.json
# regardless of the shell working directory.
CMD = f'npm --prefix "{WEB}" run dev'

print('Step 5: Starting Next.js dev server...')
print(f'  cmd = {CMD}')
proc = subprocess.Popen(
    CMD,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    shell=True
)

# Read output lines until we spot the URL Next.js prints
detected_url = None
for _ in range(120):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print(line, end='', flush=True)
    # Next.js 15 prints:  ▲ Next.js 15.x.x  and  - Local:  http://localhost:3000
    m = re.search(r'(https?://localhost:\d+)', line)
    if m:
        detected_url = m.group(1)
        for _ in range(6):
            l2 = proc.stdout.readline()
            if l2: print(l2, end='', flush=True)
        break

print()
if detected_url:
    print('==============================================')
    print('  DOOM Web Edition is RUNNING!')
    print(f'  Open:  {detected_url}')
    print('  Click the canvas to start playing.')
    print('==============================================')
    DOOM_URL = detected_url
else:
    DOOM_URL = 'http://localhost:3000'
    print(f'Server may still be starting. Try: {DOOM_URL}')


Step 3: Starting Next.js dev server...

> web-doom@1.0.0 dev
> next dev

  â–² Next.js 14.2.35
  - Local:        http://localhost:3000

 âœ“ Starting...
 âœ“ Ready in 14.1s
 â—‹ Compiling / ...
 â¨¯ ./styles/Doom.module.css:187:1

  DOOM Web Edition is RUNNING!
  Open:  http://localhost:3000
  Click the canvas to start playing.


In [ ]:
import webbrowser, time

# DOOM_URL is set by the cell above; fall back to 3000 if run standalone
url = globals().get('DOOM_URL', 'http://localhost:3000')

time.sleep(2)   # brief pause so the server is fully ready
print(f'Opening {url} ...')
webbrowser.open(url)
print('Done — DOOM should be running in your default browser.')
print('To stop the server, interrupt the kernel (■) or restart it.')


Opening http://localhost:3000 ...
Done — DOOM should be running in your default browser.
To stop the server, interrupt the kernel (■) or restart it.
